In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

data = pd.read_csv('/kaggle/input/datasets/thuyduc/fpt-stock/FPT_stock.csv')
data.head(20)

,time,open,high,low,close,volume
0,2024-01-02 09:15:00,70.92,70.92,70.40,70.92,51200
1,2024-01-02 09:16:00,70.92,70.92,70.62,70.62,6700
2,2024-01-02 09:17:00,70.62,70.70,70.62,70.70,1200
3,2024-01-02 09:18:00,70.70,70.84,70.70,70.70,1400
4,2024-01-02 09:19:00,70.70,70.70,70.70,70.70,16500
5,2024-01-02 09:20:00,70.70,70.70,70.62,70.62,9600
6,2024-01-02 09:21:00,70.62,70.70,70.62,70.70,8100
7,2024-01-02 09:22:00,70.70,70.70,70.70,70.70,1500
8,2024-01-02 09:23:00,70.77,70.84,70.77,70.84,1400
9,2024-01-02 09:24:00,70.70,70.70,70.70,70.70,18300


In [3]:
data['time'] = pd.to_datetime(data['time'], format='%Y-%m-%d %H:%M:%S')

In [4]:
# create new feature: EMA
# this is method using pandas version (recommended to use)
def calculate_ema(prices:pd.Series, period:int) -> pd.Series:
    init_ema = prices.iloc[:period].mean()

    ema = prices.copy()
    ema.iloc[:period-1] = np.nan
    ema.iloc[period-1] = init_ema
    
    final_ema = ema.ewm(span=period, adjust=False).mean()
    
    return final_ema

# calculate_ema(data['close'], 10).head(20)

In [5]:
# create new feature: RSI

def calculate_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    # 1. Get raw deltas, gains, and losses
    delta = prices.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)

    # 2. Calculate the exact initial SMA for the first window
    # index 0 is NaN. iloc[1:period+1] captures exactly 'period' (14) actual changes.
    init_gain = gain.iloc[1:period+1].mean()
    init_loss = loss.iloc[1:period+1].mean()

    # 3. The Seeding Trick: Clear early rows and plant the SMA seed at index 'period'
    gain_seeded = gain.copy()
    gain_seeded.iloc[:period] = np.nan
    gain_seeded.iloc[period] = init_gain

    loss_seeded = loss.copy()
    loss_seeded.iloc[:period] = np.nan
    loss_seeded.iloc[period] = init_loss

    # 4. Run EWM. adjust=False ignores leading NaNs and treats the seed as the true starting point.
    avg_gain = gain_seeded.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss_seeded.ewm(alpha=1/period, adjust=False).mean()

    # 5. Calculate final RSI
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    
    return rsi

In [6]:
# calculate log return

def calculate_log_return(prices:pd.Series) -> pd.Series:
    log_return = np.log(prices / prices.shift(1))
    return log_return

In [7]:
# create pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, StandardScaler

from sklearn import set_config
set_config(transform_output='pandas', display='diagram')

standardise = ColumnTransformer(
    transformers=
    [
        ("robust_scaler", RobustScaler(), ['volume']),
        ('standard_scaler', StandardScaler(), ['open', 'high', 'low', 'close'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False, # not containing the transformer name in columns of Dataframe
    force_int_remainder_cols=False
)

In [8]:
from sklearn.model_selection import train_test_split

data = data.sort_values('time')

X = data.drop(columns=['time'])
y = data['close']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, shuffle=False)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.25, shuffle=False)

In [16]:
X_train

,open,high,low,close,volume
0,70.92,70.92,70.40,70.92,51200
1,70.92,70.92,70.62,70.62,6700
2,70.62,70.70,70.62,70.70,1200
3,70.70,70.84,70.70,70.70,1400
4,70.70,70.70,70.70,70.70,16500
...,...,...,...,...,...
73919,93.01,93.10,93.01,93.01,74100
73920,93.01,93.01,93.01,93.01,3500
73921,93.01,93.10,93.01,93.10,5300
73922,93.01,93.10,93.01,93.10,2100


In [17]:
X_val

,open,high,low,close,volume
73924,93.10,93.10,93.01,93.01,3500
73925,93.01,93.10,93.01,93.10,3900
73926,93.01,93.01,93.01,93.01,49100
73927,93.01,93.01,93.01,93.01,16300
73928,93.01,93.01,93.01,93.01,7500
...,...,...,...,...,...
98561,94.52,94.71,94.52,94.62,11900
98562,94.62,94.71,94.62,94.62,5300
98563,94.71,94.81,94.62,94.71,128900
98564,94.81,94.81,94.71,94.81,47800


In [9]:
X_test

,open,high,low,close,volume
98566,94.71,94.81,94.71,94.81,7000
98567,94.71,94.91,94.71,94.81,22300
98568,94.81,94.81,94.71,94.81,13800
98569,94.71,94.81,94.71,94.71,12200
98570,94.71,94.91,94.71,94.81,26300
...,...,...,...,...,...
131417,73.00,73.10,73.00,73.00,39100
131418,73.00,73.10,72.90,73.00,42100
131419,73.00,73.00,72.90,73.00,23500
131420,73.00,73.10,72.90,73.00,35700


In [22]:
X_train = standardise.fit_transform(X_train, y_train)
X_train

,volume,open,high,low,close
0,2.344633,-1.983699,-1.986273,-2.011679,-1.983740
1,-0.169492,-1.983699,-1.986273,-1.998712,-2.001420
2,-0.480226,-2.001378,-1.999234,-1.998712,-1.996705
3,-0.468927,-1.996663,-1.990986,-1.993996,-1.996705
4,0.384181,-1.996663,-1.999234,-1.993996,-1.996705
...,...,...,...,...,...
73919,3.638418,-0.681914,-0.679584,-0.678962,-0.681931
73920,-0.350282,-0.681914,-0.684886,-0.678962,-0.681931
73921,-0.248588,-0.681914,-0.679584,-0.678962,-0.676627
73922,-0.429379,-0.681914,-0.679584,-0.678962,-0.676627


In [23]:
X_test = standardise.transform(X_test)
X_val = standardise.transform(X_val)

In [24]:
y_train = X_train['close']
y_test = X_test['close']
y_val = X_val['close']

In [ ]:
# save learned parameters from ColumnTransformer
import joblib

# Save it
joblib.dump(standardise, '/Model/standardise_transformer.joblib')

['standardise_transformer.joblib']

In [ ]:
y_train.to_csv('/Dataset/y_train_stock_fpt.csv')
y_test.to_csv('/Dataset/y_test_stock_fpt.csv')
y_val.to_csv('/Dataset/y_val_stock_fpt.csv')

In [ ]:
X_train.to_csv('/Dataset/x_train_stock_fpt.csv')
X_test.to_csv('/Dataset/x_test_stock_fpt.csv')
X_val.to_csv('/Dataset/x_val_stock_fpt.csv')